# Laboratorium 1: Dekoratory, Deskryptory i Generatory
### Skoroszyt główny

---

## Cele Laboratorium
Celem dzisiejszych zajęć jest opanowanie zaawansowanych konstrukcji języka Python, które są niezbędne do projektowania nowoczesnej architektury aplikacji.

### System Wspomagania AI (Tutor)
W trakcie rozwiązywania zadań możesz korzystać z pomocy dedykowanego tutora AI. System oferuje 6 poziomów wsparcia:
1. **Ogólna wskazówka**: Sugestia kierunku rozwiązania.
2. **Pseudokod**: Logiczny opis algorytmu.
3. **Mały fragment kodu**: Kluczowa linia lub konstrukcja.
4. **Częściowa implementacja**: Szkielet kodu do uzupełnienia.
5. **Szczegółowe wyjaśnienie**: Analiza mechanizmu działania.
6. **Pełne rozwiązanie**: Dostępne w sytuacjach ostatecznych.

---

## 1. Dekoratory

### DEMO: Dekorator @timer
Stwórz dekorator @timer, który będzie mierzył i wyświetlał czas wykonania funkcji.

In [ ]:
import time
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        print(f"Czas wykonania {func.__name__}: {end_time - start_time:.4f} s")
        return result
    return wrapper

@timer
def example_task():
    time.sleep(0.5)
    print("Zadanie zakończone.")

example_task()

Zadanie zakończone.
Czas wykonania example_task: 0.5051 s


### Zadanie 1: Liczba elementów listy
Stwórz dekorator, który będzie odpowiedzialny za wyświetlanie liczby elementów listy, jeśli jakakolwiek lista pojawi się w parametrach funkcji dekorowanej.

**Protip:** użyj isinstance do sprawdzenia czy parametr jest listą. Pamiętaj o zachowaniu metadanych funkcji.

In [4]:
import functools

def show_list_length(func):
  @functools.wraps(func)
  def wrapper(*args, **kwargs):
    for arg in args:
      if isinstance(arg, list):
        print(f"Liczba elementów listy: {len(arg)}")
    for value in kwargs.values():
      if isinstance(value, list):
        print(f"Liczba elementów listy: {len(value)}")
    return func(*args, **kwargs)
  return wrapper

@show_list_length
def process_data(data_list, name):
    print(f"Przetwarzanie {name}")
process_data([1, 2, 3, 4, 5], "moja lista.")

Liczba elementów listy: 5
Przetwarzanie moja lista.


### Zadanie 2: Logowanie do pliku
Stwórz dekorator, który będzie zapisywał w pliku *.log nazwę funkcji dekorowanej, datę oraz długość wykonania. Nazwa pliku będzie podana jako argument dekoratora.

**Protip:** użyj biblioteki datetime. Pamiętaj o tym, żeby dekoratory przyjęły metadanych funkcji dekorującej.

In [8]:
import functools
from datetime import datetime
import time

def logger(filename):
  def decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
      start_time = time.perf_counter()
      result = func(*args, **kwargs)
      end_time = time.perf_counter()
      execution_time = end_time - start_time
      current_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
      with open(filename, "a", encoding="utf-8") as file:
        file.write(
            f"Nazwa funkcji: {func.__name__}, "
            f"Data: {current_date}, "
            f"Czas wykonania: {execution_time:.6f} s\n"
        )
      return result
    return wrapper
  return decorator

@logger("test.log")
def test_function():
  time.sleep(1)
  print("Funkcja testowa została wykonana")
test_function()

Funkcja testowa została wykonana


In [9]:
with open("test.log", "r", encoding="utf-8") as file:
    print(file.read())

Nazwa funkcji: test_function, Data: 2026-03-21 18:48:40, Czas wykonania: 1.000271 s



---
## 2. Deskryptory

### DEMO: Walidator e-mail klasy Student
Stwórz deskryptor, który będzie działał jako walidator email klasy Student. Klasa Student zawiera pola imie, nazwisko i email. Deskryptor ten powinien sprawdzać poprawność danych wprowadzanych podczas tworzenia lub modyfikowania instancji Student.

In [ ]:
class EmailValidator:
    def __set_name__(self, owner, name):
        self.name = name

    def __set__(self, instance, value):
        if "@" not in value:
            raise ValueError(f"Błędny format adresu email: {value}")
        instance.__dict__[self.name] = value

class Student:
    email = EmailValidator()

    def __init__(self, imie, nazwisko, email):
        self.imie = imie
        self.nazwisko = nazwisko
        self.email = email

try:
    s = Student("Jan", "Kowalski", "jan.kowalski@wsei.edu.pl")
    print(f"Utworzono studenta: {s.email}")
    # s.email = "invalid_at_email" # Powinno rzucić błąd
except ValueError as e:
    print(e)

Utworzono studenta: jan.kowalski@wsei.edu.pl


### Zadanie 3: Rejestrowanie dostępu
Stwórz klasę Uzytkownik. Klasa powinna zawierać atrybuty imie i wiek. Opracuj deskryptor, który będzie rejestrował dostęp do tych atrybutów za pomocą logowania. Deskryptor powinien logować informacje o odczycie (__get__) oraz zapisie (__set__) wartości atrybutu.

In [10]:
class AccessLogger:
  def __init__(self, name):
    self.name = name
    self.private_name = "_" + name

  def __get__(self, instance, owner):
    if instance is None:
      return self
    value = getattr(instance, self.private_name)
    print(f"Odczytano {self.name}: {value}")
    return value

  def __set__(self, instance, value):
    print(f"Zapisano {self.name}: {value}")
    setattr(instance, self.private_name, value)

class Uzytkownik:
  imie = AccessLogger("imie")
  wiek = AccessLogger("wiek")

  def __init__(self, imie, wiek):
    self.imie = imie
    self.wiek = wiek

uzytkownik = Uzytkownik("Emilia", 22)
print(uzytkownik.imie)
print(uzytkownik.wiek)

uzytkownik.imie = "Katarzyna"
uzytkownik.wiek = 23

print(uzytkownik.imie)
print(uzytkownik.wiek)

Zapisano imie: Emilia
Zapisano wiek: 22
Odczytano imie: Emilia
Emilia
Odczytano wiek: 22
22
Zapisano imie: Katarzyna
Zapisano wiek: 23
Odczytano imie: Katarzyna
Katarzyna
Odczytano wiek: 23
23


---
## 3. Generatory i Iteratory

### DEMO: Generator Fibonacciego
Napisz klasę, która będzie implementowała generator ciągu Fibonacciego za pomocą metod magicznych __iter__() i __next__().

In [ ]:
class FibonacciGenerator:
    def __init__(self, limit):
        self.limit = limit
        self.a, self.b = 0, 1
        self.count = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.count >= self.limit:
            raise StopIteration

        result = self.a
        self.a, self.b = self.b, self.a + self.b
        self.count += 1
        return result

fib = FibonacciGenerator(10)
print(list(fib))

### Zadanie 4: Generator ciągu Collatza
Opracuj generator ciągu Collatza. Dla liczby naturalnej n, jeśli n jest parzyste, dziel przez 2; jeśli n jest nieparzyste, pomnóż przez 3 i dodaj 1, zaczynając od określonej liczby początkowej, aż do osiągnięcia wartości 1.

In [11]:
def collatz_generator(n):
  while n != 1:
    yield n
    if n % 2 == 0:
      n //= 2
    else:
      n = 3 * n + 1
  yield 1

# Test:
for status in collatz_generator(10):
  print(status)

10
5
16
8
4
2
1


---

## Zadania do zrobienia w domu

Poniższe zadania stanowią rozszerzenie materiału i są przeznaczone dla osób chcących zgłębić temat zaawansowanych konstrukcji języka Python.

### Zadanie dodatkowe 1: Dekorator z autoryzacją
Stwórz dekorator `@require_role(role)`, który przyjmuje nazwę wymaganej roli jako argument. Dekorator powinien sprawdzać, czy w globalnym słowniku `current_user` klucz `role` jest zgodny z wymaganym. Jeśli nie, rzuć `PermissionError`.

In [13]:
import functools

current_user = {"username": "admin", "role": "superuser"}

def require_role(role):
  def decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
      if current_user.get("role") != role:
        raise PermissionError(
            f"Brak uprawnień. Wymagana rola: {role}, aktualna rola: {current_user.get('role')}"
        )
      return func(*args, **kwargs)
    return wrapper
  return decorator

@require_role("superuser")
def panel_admina():
  print("To jest funkcja dla administratorów.")

panel_admina()

To jest funkcja dla administratorów.


### Zadanie dodatkowe 2: Deskryptor z walidacją typu
Stwórz deskryptor `Typed`, który przyjmuje typ danych (np. `int`, `str`) w konstruktorze. Deskryptor powinien upewnić się, że zapisywana wartość jest tego typu. Jeśli nie, rzuć `TypeError`.

In [15]:
class Typed:
  def __init__(self, name, expected_type):
    self.name = name
    self.private_name = "_" + name
    self.expected_type = expected_type

  def __get__(self, instance, owner):
    if instance is None:
      return self
    return getattr(instance, self.private_name)

  def __set__(self, instance, value):
    if not isinstance(value, self.expected_type):
      raise TypeError(
          f"Atrybut '{self.name}' musi być typu {self.expected_type.__name__}"
      )
    setattr(instance, self.private_name, value)

class Osoba:
  imie = Typed("imie", str)
  wiek = Typed("wiek", int)

  def __init__(self, imie, wiek):
    self.imie = imie
    self.wiek = wiek

o1 = Osoba("Emilia", 22)
print(o1.imie)
print(o1.wiek)


Emilia
22


### Zadanie dodatkowe 3: Nieskończony generator liczb pierwszych
Opracuj generator `prime_generator`, który zwraca kolejne liczby pierwsze. Następnie użyj wyrażenia generatorowego, aby stworzyć iterator zwracający tylko te liczby pierwsze, które kończą się cyfrą 7.

In [19]:
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n ** 0.5) + 1):
        if n % i == 0:
            return False
    return True


def prime_generator():
    n = 2
    while True:
        if is_prime(n):
            yield n
        n += 1


prime_ending_7 = (p for p in prime_generator() if p % 10 == 7)

for _ in range(10):
    print(next(prime_ending_7))

7
17
37
47
67
97
107
127
137
157
